# 04 CNN Training

This notebook trains and evaluates deep learning models for sleep-stage classification using PyTorch-ready epoch tensors. It includes the Single-Epoch CNN, Temporal-Context CNN, Many-to-One CNN-GRU, Many-to-Many CNN-GRU, Multiscale Residual CNN-MLP Fusion model (MSResCNN-MLP), and MSResCNN-MLP embedding followed by a TCN head with $n$ input epochs ($n$-epoch MSResCNN-MLP-TCN). The notebook begins with tensor shape, preprocessing, and participant-leakage checks, then runs guarded validation-only training workflows through reusable `src.train` utilities. The held-out test split is not evaluated here.

This setup cell imports the reusable data and training utilities, resolves paths whether the notebook is run from the repository root or the `notebooks/` directory, and defines a small debug configuration for the initial smoke run. The expected output is a single Boolean indicating whether local raw data and `data/interim/epoch_index.csv` are available. If it is `False`, later cells skip cleanly instead of failing on missing local DREAMT artifacts. Long-running training cells remain guarded so routine execution does not accidentally launch full experiments.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import torch

from src.error_analysis import class_prior_correction_sweep
from src.data import (
    DEFAULT_EPOCH_INDEX_PATH,
    DEFAULT_FEATURE_PREPROCESSING_METADATA_PATH,
    DEFAULT_PARTICIPANT_ARRAY_CACHE_DIR,
    DEFAULT_PREPROCESSING_METADATA_PATH,
    DEFAULT_RAW_DATA_DIR,
    DEFAULT_STAGE15_EMBEDDING_DIR,
    DEFAULT_TRAIN_FEATURES_PATH,
    DEFAULT_VALIDATION_FEATURES_PATH,
    DreamtContextDataset,
    DreamtEpochDataset,
    DreamtSequenceDataset,
    check_epoch_split_leakage,
    fit_normalization_stats,
    load_preprocessing_metadata,
    save_preprocessing_metadata,
)
from src.train import (
    DEFAULT_STAGE8_OUTPUT_DIR,
    DEFAULT_STAGE9_OUTPUT_DIR,
    DEFAULT_STAGE10_OUTPUT_DIR,
    DEFAULT_STAGE11_OUTPUT_DIR,
    DEFAULT_STAGE11_LOSS_OUTPUT_DIR,
    DEFAULT_STAGE12_OUTPUT_DIR,
    DEFAULT_STAGE14_OUTPUT_DIR,
    DEFAULT_STAGE14_WEIGHTED_OUTPUT_DIR,
    DEFAULT_STAGE15_OUTPUT_DIR,
    DEFAULT_STAGE15_REPLICATION_OUTPUT_DIR,
    DEFAULT_STAGE16_OUTPUT_DIR,
    DEFAULT_STAGE16_REPLICATION_OUTPUT_DIR,
    TrainConfig,
    build_stage9_screening_configs,
    build_stage10_comparison_configs,
    build_stage11_loss_comparison_configs,
    build_stage11_sequence_configs,
    build_stage12_many_to_many_configs,
    build_stage14_fusion_config,
    build_stage14_weighted_followup_config,
    build_stage15_temporal_tcn_config,
    build_stage15_seed_replication_configs,
    build_stage16_temporal_tcn_config,
    build_stage16_seed_replication_configs,
    build_train_validation_datasets,
    class_counts_from_loader,
    load_train_config,
    run_tiny_overfit_test,
    run_stage9_experiments,
    run_stage10_experiments,
    run_stage11_experiments,
    run_stage12_experiments,
    run_stage14_experiment,
    run_stage15_experiment,
    run_stage15_seed_replications,
    run_stage16_experiment,
    run_stage16_seed_replications,
    train_model,
)

CHANNELS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
BATCH_SIZE = 16
DEBUG_PARTICIPANTS = 3
EPOCHS = 1

raw_dir = repo_root / DEFAULT_RAW_DATA_DIR
epoch_index_path = repo_root / DEFAULT_EPOCH_INDEX_PATH
metadata_path = repo_root / DEFAULT_PREPROCESSING_METADATA_PATH
array_cache_dir = repo_root / DEFAULT_PARTICIPANT_ARRAY_CACHE_DIR
train_feature_path = repo_root / DEFAULT_TRAIN_FEATURES_PATH
validation_feature_path = repo_root / DEFAULT_VALIDATION_FEATURES_PATH
feature_metadata_path = repo_root / DEFAULT_FEATURE_PREPROCESSING_METADATA_PATH
output_dir = repo_root / DEFAULT_STAGE8_OUTPUT_DIR
stage9_output_dir = repo_root / DEFAULT_STAGE9_OUTPUT_DIR
stage10_output_dir = repo_root / DEFAULT_STAGE10_OUTPUT_DIR
stage11_output_dir = repo_root / DEFAULT_STAGE11_OUTPUT_DIR
stage11_loss_output_dir = repo_root / DEFAULT_STAGE11_LOSS_OUTPUT_DIR
stage12_output_dir = repo_root / DEFAULT_STAGE12_OUTPUT_DIR
stage14_output_dir = repo_root / DEFAULT_STAGE14_OUTPUT_DIR
stage14_weighted_output_dir = repo_root / DEFAULT_STAGE14_WEIGHTED_OUTPUT_DIR
stage15_output_dir = repo_root / DEFAULT_STAGE15_OUTPUT_DIR
stage15_replication_output_dir = repo_root / DEFAULT_STAGE15_REPLICATION_OUTPUT_DIR
stage16_output_dir = repo_root / DEFAULT_STAGE16_OUTPUT_DIR
stage16_replication_output_dir = repo_root / DEFAULT_STAGE16_REPLICATION_OUTPUT_DIR
stage15_embedding_dir = repo_root / DEFAULT_STAGE15_EMBEDDING_DIR
stage8_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=output_dir,
    channels=CHANNELS,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    max_train_participants=DEBUG_PARTICIPANTS,
    max_val_participants=DEBUG_PARTICIPANTS,
)

artifacts_available = raw_dir.exists() and epoch_index_path.exists()
stage14_artifacts_available = (
    artifacts_available
    and train_feature_path.exists()
    and validation_feature_path.exists()
)
artifacts_available


## Build Single-Epoch Datasets

This section builds the PyTorch-ready epoch tensors used by the Single-Epoch CNN. The first cell fits streaming mean imputation and per-channel standardization metadata using training epochs only, then saves the metadata for reuse. Validation epochs are transformed with the saved training metadata, but they are not used to fit preprocessing statistics. The test split is intentionally not loaded in this notebook. Expected output is either no displayed output when local artifacts are present, or a skip message when raw files or `data/interim/epoch_index.csv` are unavailable.

In [ ]:
if artifacts_available:
    train_unscaled = DreamtEpochDataset(
        raw_dir=raw_dir,
        epoch_index=epoch_index_path,
        split="train",
        channels=CHANNELS,
        max_participants=DEBUG_PARTICIPANTS,
    )
    stats = fit_normalization_stats(train_unscaled)
    save_preprocessing_metadata(stats, metadata_path)
else:
    print("Skipping dataset construction because local raw files or epoch_index.csv are absent.")


This cell reloads the saved preprocessing metadata, applies it to the training and validation datasets, checks for participant-level leakage within each dataset, and constructs the corresponding DataLoaders. The printed `x` batch shape should be `(batch, channels, timepoints)`, while `y` should contain one integer sleep-stage label per epoch. The participant counts confirm the debug subset size, and the metadata channels should match the configured model channels. The test split is intentionally not loaded.

In [ ]:
if artifacts_available:
    stats = load_preprocessing_metadata(metadata_path)
    train_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    val_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="validation", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    check_epoch_split_leakage(train_ds.epoch_index)
    check_epoch_split_leakage(val_ds.epoch_index)
    loaders = {
        "train": torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
        "validation": torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False),
    }
    x_batch, y_batch = next(iter(loaders["train"]))
    print("train batch:", tuple(x_batch.shape), x_batch.dtype, tuple(y_batch.shape), y_batch.dtype)
    print("participants:", {"train": len(train_ds.participants), "validation": len(val_ds.participants)})
    print("metadata channels:", stats["channels"])


## Temporal Context And Sequence Shape Checks

This section performs lightweight shape checks for the datasets used by the Temporal-Context CNN and CNN-GRU sequence models. These checks verify that neighboring-epoch windows can be formed without crossing participant boundaries and that the resulting tensor shapes match the intended model inputs. Expected output is one example temporal-context item and one example sequence item when the debug subset contains enough consecutive training epochs.

In [ ]:
if artifacts_available:
    context_ds = DreamtContextDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, context_radius=2, max_participants=DEBUG_PARTICIPANTS)
    sequence_ds = DreamtSequenceDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, sequence_length=5, label_mode="many_to_one", target_position="center", max_participants=DEBUG_PARTICIPANTS)
    if len(context_ds):
        x_context, y_context = context_ds[0]
        print("context item:", tuple(x_context.shape), y_context.item())
    if len(sequence_ds):
        x_sequence, y_sequence = sequence_ds[0]
        print("sequence item:", tuple(x_sequence.shape), y_sequence.item())


## Tiny Overfit Smoke Test

This cell runs a tiny repeated-batch overfit test on training data only. Its purpose is to verify that the model, loss, optimizer, tensor dtypes, and device handling can reduce training loss before running validation-monitored experiments. The expected output is the first and last loss; the last value should usually be lower than the first. This is a plumbing check, not a model-selection result.

In [ ]:
if artifacts_available:
    overfit_history = run_tiny_overfit_test(train_ds, stage8_config)
    print("loss first/last:", round(overfit_history["loss"].iloc[0], 4), round(overfit_history["loss"].iloc[-1], 4))


## Single-Epoch CNN Training

This cell trains the Single-Epoch CNN and monitors validation metrics after each epoch. In the default debug configuration, it runs for one epoch on a small participant subset, so the goal is to confirm that training completes and expected artifacts are produced rather than to assess final performance. Expected outputs include the training-history table, the best epoch, and an output directory containing history, validation metrics, a confusion matrix, plots, and checkpoints. The validation split is used for monitoring; the test split remains untouched.

In [ ]:
if artifacts_available:
    training_result = train_model(loaders["train"], loaders["validation"], stage8_config)
    display(training_result.history)
    print("best epoch:", training_result.best_epoch)
    print("outputs:", training_result.output_dir)


## Single-Epoch CNN Training-Choice Experiments

This section keeps the model family fixed to the Single-Epoch CNN and compares basic training choices using validation macro F1 as the primary selection metric. The first cell defines the base configuration and expands it into a controlled screening grid that uses train-only class-weighted loss, varies learning rate and dropout, and keeps weight decay fixed at `0.0`. Expected output is the number of configured runs. These configurations use only the training and validation splits; the test split remains untouched.

In [ ]:
stage9_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage9_output_dir,
    channels=CHANNELS,
    batch_size=32,
    epochs=40,
    patience=10,
    train_eval_interval=None,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
)

stage9_screening_configs = build_stage9_screening_configs(
    base_config=stage9_base_config,
    output_dir=stage9_output_dir,
    learning_rates=(1e-4, 3e-4, 1e-3),
    dropouts=(0.0, 0.10),
    weight_decays=(0.0,),
    class_weighting_options=(True,),
    batch_sizes=(32,),
)
len(stage9_screening_configs)


The configured grid has 6 runs. For a quick local dry run, slice `stage9_screening_configs` before calling `run_stage9_experiments`; for the main result, run the full list. The next cell is guarded by `RUN_STAGE9_EXPERIMENTS = False` so routine notebook execution does not accidentally launch a long training sweep. When enabled, expected output is a validation summary sorted by macro F1, plus a results directory containing per-run histories, validation metrics, confusion matrices, checkpoints, and aggregate summary files.

In [ ]:
RUN_STAGE9_EXPERIMENTS = False

if artifacts_available and RUN_STAGE9_EXPERIMENTS:
    stage9_summary = run_stage9_experiments(
        stage9_screening_configs,
        output_dir=stage9_output_dir,
    )
    display(stage9_summary.sort_values("macro_f1", ascending=False).head(10))
    print("outputs:", stage9_output_dir)
else:
    print("Stage 9 experiments are configured but not run in this notebook execution.")


## Temporal-Context CNN Comparison

This section asks whether neighboring epochs improve validation performance relative to the Single-Epoch CNN. The comparison is deliberately conservative: it keeps the CNN architecture and broad training defaults fixed, tests `context_radius=2` and `context_radius=5`, and pairs each Temporal-Context CNN with a center-only CNN trained and evaluated on the same context-eligible center epochs. This makes the validation comparison between center-only input and center-plus-neighbor input, rather than between different validation epoch sets. The test split remains untouched.


In [ ]:
stage10_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage10_output_dir,
    channels=CHANNELS,
    batch_size=32,
    epochs=40,
    patience=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    dropout=0.0,
    class_weighting=True,
    train_eval_interval=None,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
)

stage10_comparison_configs = build_stage10_comparison_configs(
    base_config=stage10_base_config,
    output_dir=stage10_output_dir,
    context_radii=(2, 5),
)
[(config.model_name, config.context_radius, config.comparison_context_radius) for config in stage10_comparison_configs]


The next cell is guarded by `RUN_STAGE10_EXPERIMENTS = False` so routine notebook execution does not accidentally launch the comparison. When enabled, it trains four validation-only runs: a center-only Single-Epoch CNN and a Temporal-Context CNN for `context_radius=2`, followed by the same pair for `context_radius=5`. The output summary includes validation macro F1, balanced accuracy, class-level metrics, the paired context radius, and the number of matched center epochs used for each comparison. Broader hyperparameter tuning is intentionally deferred to separate training experiments.

In [ ]:
RUN_STAGE10_EXPERIMENTS = False

if artifacts_available and RUN_STAGE10_EXPERIMENTS:
    stage10_summary = run_stage10_experiments(
        stage10_comparison_configs,
        output_dir=stage10_output_dir,
    )
    display(stage10_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage10_output_dir)
else:
    print("Stage 10 experiments are configured but not run in this notebook execution.")


## Many-to-One CNN-GRU Sequence Comparison

This section trains the first recurrent deep learning model: the Many-to-One CNN-GRU. Each epoch in a consecutive sequence is encoded by the CNN trunk, the bidirectional GRU models temporal context around the center epoch, and the classifier predicts the center epoch label. The initial run uses `sequence_length=5` with class-weighted cross entropy. The test split remains untouched.


In [ ]:
stage11_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage11_output_dir,
    channels=CHANNELS,
    batch_size=16,
    epochs=25,
    patience=5,
    learning_rate=3e-4,
    weight_decay=1e-4,
    filters=(16, 32, 64),
    kernel_size=31,
    dropout=0.0,
    class_weighting=True,
    max_grad_norm=1.0,
    train_eval_interval=None,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
    sequence_stride=1,
    sequence_label_mode="many_to_one",
    sequence_target_position="center",
    gru_hidden_size=64,
    gru_num_layers=1,
    gru_dropout=0.0,
    gru_bidirectional=True,
)

stage11_sequence_configs = build_stage11_sequence_configs(
    base_config=stage11_base_config,
    output_dir=stage11_output_dir,
    sequence_lengths=(5,),
)
[(config.model_name, config.sequence_length, config.sequence_target_position, config.class_weighting) for config in stage11_sequence_configs]


The next cell is guarded by `RUN_STAGE11_EXPERIMENTS = False` so routine notebook execution does not accidentally launch the sequence model training run. When enabled, it trains the initial Many-to-One CNN-GRU with `sequence_length=5`. The output summary includes validation macro F1, balanced accuracy, class-level metrics, and the number of sequence examples used for training and validation.

In [ ]:
RUN_STAGE11_EXPERIMENTS = False

if artifacts_available and RUN_STAGE11_EXPERIMENTS:
    stage11_summary = run_stage11_experiments(
        stage11_sequence_configs,
        output_dir=stage11_output_dir,
    )
    display(stage11_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage11_output_dir)
else:
    print("Stage 11 experiments are configured but not run in this notebook execution.")


## Many-to-One CNN-GRU Class-Prior Diagnostic

This no-training diagnostic tests whether the completed class-weighted Many-to-One CNN-GRU learned useful class separation but produced an overly REM-heavy decision boundary. It adjusts saved validation probabilities using powered training priors for `alpha` values from `0` through `1`, then saves validation metrics for each correction. These are validation-set development diagnostics, not final unbiased performance estimates.


In [ ]:
stage11_prior_correction_summary = pd.DataFrame()
stage11_summary_path = stage11_output_dir / "experiment_summary.csv"

if stage11_summary_path.exists():
    completed_stage11 = pd.read_csv(stage11_summary_path).sort_values(
        "macro_f1", ascending=False
    )
    completed_run_dir = Path(completed_stage11.iloc[0]["output_dir"])
    prediction_path = completed_run_dir / "validation_epoch_predictions.csv"
    config_path = completed_run_dir / "config.json"
    if prediction_path.exists() and config_path.exists():
        completed_config = load_train_config(config_path=config_path)
        completed_datasets = build_train_validation_datasets(completed_config)
        count_loader = torch.utils.data.DataLoader(
            completed_datasets["train"], batch_size=completed_config.batch_size
        )
        stage11_train_counts = class_counts_from_loader(count_loader)
        stage11_predictions = pd.read_csv(prediction_path)
        stage11_prior_correction_summary = class_prior_correction_sweep(
            stage11_predictions,
            stage11_train_counts,
            alphas=(0.0, 0.25, 0.5, 0.75, 1.0),
        )
        stage11_prior_correction_summary.to_csv(
            stage11_output_dir / "prior_correction_summary.csv", index=False
        )

display(stage11_prior_correction_summary)


## Many-to-One CNN-GRU Loss-Weighting Follow-Up

This follow-up keeps the `sequence_length=5` bidirectional Many-to-One CNN-GRU fixed and compares two loss choices: unweighted cross entropy and square-root inverse-frequency weighting. Both runs use a 15-epoch cap with patience 4 and write to a separate output directory so the initial CNN-GRU results are preserved.


In [ ]:
stage11_loss_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage11_loss_output_dir,
    channels=CHANNELS,
    batch_size=16,
    epochs=15,
    patience=4,
    learning_rate=3e-4,
    weight_decay=1e-4,
    filters=(16, 32, 64),
    kernel_size=31,
    dropout=0.0,
    max_grad_norm=1.0,
    train_eval_interval=None,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
    sequence_stride=1,
    sequence_label_mode="many_to_one",
    sequence_target_position="center",
    gru_hidden_size=64,
    gru_num_layers=1,
    gru_dropout=0.0,
    gru_bidirectional=True,
)

stage11_loss_configs = build_stage11_loss_comparison_configs(
    base_config=stage11_loss_base_config,
    output_dir=stage11_loss_output_dir,
    sequence_length=5,
)
[(config.model_name, config.class_weighting, config.class_weight_power) for config in stage11_loss_configs]


In [ ]:
RUN_STAGE11_LOSS_EXPERIMENTS = False

if artifacts_available and RUN_STAGE11_LOSS_EXPERIMENTS:
    stage11_loss_summary = run_stage11_experiments(
        stage11_loss_configs,
        output_dir=stage11_loss_output_dir,
    )
    display(stage11_loss_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage11_loss_output_dir)
else:
    print("Stage 11 loss experiments are configured but not run.")


## Many-To-Many CNN-GRU Aggregation

This section trains CNN-GRU models that predict a label for every epoch in each input sequence. Because overlapping sequences produce multiple probability predictions for most sleep epochs, validation aggregates those probabilities back to one prediction per epoch before model selection. The default setup compares sequence_length=5 and sequence_length=11, uses class-weighted cross entropy, weights each sequence-position loss by `1 / number_of_times_that_sleep_epoch_appears`, and evaluates both uniform and center-weighted probability averaging from the same trained checkpoint. The test split remains untouched.


In [ ]:
stage12_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage12_output_dir,
    channels=CHANNELS,
    batch_size=16,
    epochs=12,
    patience=8,
    learning_rate=3e-4,
    weight_decay=1e-4,
    filters=(16, 32, 64),
    kernel_size=31,
    dropout=0.0,
    class_weighting=True,
    max_grad_norm=1.0,
    train_eval_interval=None,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
    sequence_stride=1,
    sequence_label_mode="many_to_many",
    sequence_target_position="center",
    sequence_loss_weighting="inverse_epoch_coverage",
    gru_hidden_size=64,
    gru_num_layers=1,
    gru_dropout=0.0,
    gru_bidirectional=False,
)

stage12_many_to_many_configs = build_stage12_many_to_many_configs(
    base_config=stage12_base_config,
    output_dir=stage12_output_dir,
    sequence_lengths=(5, 11),
    aggregation_methods=("uniform", "center_weighted"),
)
[(config.model_name, config.sequence_length, config.sequence_aggregation, config.sequence_extra_aggregations, config.sequence_loss_weighting) for config in stage12_many_to_many_configs]


The next cell is guarded by `RUN_STAGE12_EXPERIMENTS = False` so routine notebook execution does not accidentally launch the aggregation experiments. When enabled, it trains one Many-to-Many CNN-GRU per sequence length, then reports one summary row per aggregation method. The output includes raw sequence-position metrics, aggregated per-epoch metrics, per-position validation predictions, aggregated epoch-level validation predictions, confusion matrices, training histories, and checkpoints.


In [ ]:
RUN_STAGE12_EXPERIMENTS = False

if artifacts_available and RUN_STAGE12_EXPERIMENTS:
    stage12_summary = run_stage12_experiments(
        stage12_many_to_many_configs,
        output_dir=stage12_output_dir,
    )
    display(stage12_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage12_output_dir)
else:
    print("Stage 12 experiments are configured but not run in this notebook execution.")


## Multiscale Residual CNN-MLP Fusion

This section trains the Multiscale Residual CNN-MLP Fusion model (MSResCNN-MLP). The raw-signal branch uses three convolutional scales, GroupNorm residual blocks, and retained temporal bins to encode each epoch. A compact MLP encodes the engineered epoch-level features, and the two embeddings are fused to produce one three-class sleep-stage prediction per epoch. Raw-signal and engineered-feature preprocessing are fit using training rows only; validation data reuse the saved training metadata. The fixed run uses unweighted cross entropy with label smoothing, validation macro F1 checkpointing, gradient clipping, and full train-set evaluation at the scheduled final epoch or early-stopping epoch.

In [ ]:
stage14_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage14_output_dir,
    channels=CHANNELS,
    train_feature_path=train_feature_path,
    validation_feature_path=validation_feature_path,
    feature_preprocessing_metadata_path=feature_metadata_path,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
)

stage14_config = build_stage14_fusion_config(
    base_config=stage14_base_config,
    output_dir=stage14_output_dir,
)
stage14_config


The launch cell is deliberately guarded by `RUN_STAGE14_EXPERIMENT = False` so routine notebook execution does not accidentally start training. Enabling it runs one fixed MSResCNN-MLP configuration; there is no sweep. Expected outputs include the run configuration, train and validation histories, macro-F1-selected checkpoints, validation predictions, a confusion matrix, and `results/stage14_multiscale_fusion_cnn/experiment_summary.csv`.

In [ ]:
RUN_STAGE14_EXPERIMENT = False

if stage14_artifacts_available and RUN_STAGE14_EXPERIMENT:
    stage14_summary = run_stage14_experiment(
        stage14_config,
        output_dir=stage14_output_dir,
    )
    display(stage14_summary)
    print("outputs:", stage14_output_dir)
else:
    print("Stage 14 is configured but not run in this notebook execution.")


## MSResCNN-MLP Square-Root-Weighted Follow-Up

The completed unweighted `MSResCNN-MLP` run exceeded the macro F1 target but favored Non-REM and achieved weak REM recall. This controlled follow-up changes only the loss weighting by using powered inverse-frequency weights with `class_weight_power=0.5`. The architecture, data, learning rate, label smoothing, random seed, epoch limit, and early stopping settings remain fixed. Results are written to a separate directory so the original run remains intact.

In [ ]:
stage14_weighted_config = build_stage14_weighted_followup_config(
    base_config=stage14_base_config,
    output_dir=stage14_weighted_output_dir,
)
(
    stage14_weighted_config.model_name,
    stage14_weighted_config.class_weighting,
    stage14_weighted_config.class_weight_power,
    stage14_weighted_config.epochs,
    stage14_weighted_config.patience,
)


Set `RUN_STAGE14_WEIGHTED_FOLLOWUP = True` to launch this one follow-up run. Leave `RUN_STAGE14_EXPERIMENT = False` above so the completed unweighted MSResCNN-MLP model is not retrained. Compare macro F1 and all three class F1 scores before deciding whether either fusion model is suitable for final held-out test evaluation.

In [ ]:
RUN_STAGE14_WEIGHTED_FOLLOWUP = False

if stage14_artifacts_available and RUN_STAGE14_WEIGHTED_FOLLOWUP:
    stage14_weighted_summary = run_stage14_experiment(
        stage14_weighted_config,
        output_dir=stage14_weighted_output_dir,
    )
    display(stage14_weighted_summary)
    print("outputs:", stage14_weighted_output_dir)
else:
    print("Stage 14 weighted follow-up is configured but not run.")


## 31-epoch MSResCNN-MLP-TCN

This section tests whether explicit temporal context can improve on the weighted MSResCNN-MLP model without changing its epoch representation. The best weighted MSResCNN-MLP checkpoint is frozen and used once to cache a 160-dimensional embedding for every training and validation epoch. A compact non-causal temporal convolutional network then predicts one label for every position in each 31-epoch window. Dilations `(1, 2, 4, 8)` cover the local sequence, inverse-coverage weights reduce the effect of heavily overlapped epochs, and overlapping probabilities are aggregated back to one prediction per epoch. Center-weighted aggregation is primary because middle positions have the fullest context; uniform aggregation is retained as a secondary diagnostic.

In [ ]:
stage14_weighted_summary_path = stage14_weighted_output_dir / "experiment_summary.csv"
stage15_encoder_checkpoint_path = None

if stage14_weighted_summary_path.exists():
    completed_stage14_weighted = pd.read_csv(stage14_weighted_summary_path)
    completed_stage14_weighted = completed_stage14_weighted.sort_values(
        ["macro_f1", "balanced_accuracy"],
        ascending=[False, False],
    )
    stage15_source_run_dir = Path(
        completed_stage14_weighted.iloc[0]["output_dir"]
    )
    stage15_encoder_checkpoint_path = (
        stage15_source_run_dir / "checkpoints" / "best.pt"
    )

stage15_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage15_output_dir,
    channels=CHANNELS,
    train_feature_path=train_feature_path,
    validation_feature_path=validation_feature_path,
    feature_preprocessing_metadata_path=feature_metadata_path,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
)

stage15_config = None
if stage15_encoder_checkpoint_path is not None and stage15_encoder_checkpoint_path.exists():
    stage15_config = build_stage15_temporal_tcn_config(
        stage15_encoder_checkpoint_path,
        base_config=stage15_base_config,
        output_dir=stage15_output_dir,
        embedding_dir=stage15_embedding_dir,
    )

stage15_config


The first enabled 31-epoch MSResCNN-MLP-TCN run exports the frozen embeddings before training; later reruns reuse them when the checkpoint and source artifacts are unchanged. The long experiment is guarded by `RUN_STAGE15_EXPERIMENT = False`. Leave both MSResCNN-MLP launch flags disabled. Expected outputs include the embedding cache under `data/processed/stage15_embeddings/`, per-run histories and checkpoints, center-weighted and uniform aggregated validation predictions, and `results/stage15_temporal_fusion_tcn/experiment_summary.csv`.


In [ ]:
RUN_STAGE15_EXPERIMENT = False

if stage15_config is not None and RUN_STAGE15_EXPERIMENT:
    stage15_summary = run_stage15_experiment(
        stage15_config,
        output_dir=stage15_output_dir,
    )
    display(stage15_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage15_output_dir)
else:
    print("Stage 15 is configured but not run in this notebook execution.")


## 31-epoch MSResCNN-MLP-TCN Seed Replication and Equal-Weight Ensemble

This section preserves the completed seed-42 31-epoch MSResCNN-MLP-TCN checkpoint, repeats the same 31-epoch MSResCNN-MLP-TCN with predetermined seeds 43 and 44, and then averages the three center-weighted epoch probability tables equally. Seed 42 is not retrained. The replication summary reports each seed separately, the across-seed mean and sample standard deviation, and the ensemble result; no best-seed selection is performed.

In [ ]:
stage15_summary_path = stage15_output_dir / "experiment_summary.csv"
stage15_reference_run_dir = None

if stage15_summary_path.exists():
    completed_stage15 = pd.read_csv(stage15_summary_path)
    completed_stage15_primary = completed_stage15[
        completed_stage15["aggregation_method"] == "center_weighted"
    ].sort_values(
        ["macro_f1", "balanced_accuracy"],
        ascending=[False, False],
    )
    if not completed_stage15_primary.empty:
        stage15_reference_run_dir = Path(
            completed_stage15_primary.iloc[0]["output_dir"]
        )

stage15_replication_configs = []
if stage15_config is not None and stage15_reference_run_dir is not None:
    stage15_replication_configs = build_stage15_seed_replication_configs(
        stage15_config,
        output_dir=stage15_replication_output_dir,
        seeds=(43, 44),
    )

[(config.random_seed, config.sequence_length) for config in stage15_replication_configs]


Set `RUN_STAGE15_SEED_REPLICATIONS = True` to train seeds 43 and 44 and create the equal-weight seed-42/43/44 31-epoch MSResCNN-MLP-TCN ensemble. Existing completed replication runs are reused when their expected prediction artifacts are present. Outputs are written under `results/stage15_temporal_fusion_tcn_seed_replication/`, leaving the original seed-42 run untouched.

In [ ]:
RUN_STAGE15_SEED_REPLICATIONS = False

if (
    stage15_reference_run_dir is not None
    and stage15_replication_configs
    and RUN_STAGE15_SEED_REPLICATIONS
):
    stage15_seed_ensemble_summary = run_stage15_seed_replications(
        stage15_replication_configs,
        reference_run_dir=stage15_reference_run_dir,
        output_dir=stage15_replication_output_dir,
    )
    display(stage15_seed_ensemble_summary)
    print("outputs:", stage15_replication_output_dir)
else:
    print("Stage 15 seed replications are configured but not run.")


## 61-Epoch MSResCNN-MLP-TCN

This section is a controlled temporal-context follow-up to the 31-epoch MSResCNN-MLP-TCN. It reuses the same frozen square-root-weighted MSResCNN-MLP encoder, cached 160-dimensional embeddings, TCN architecture, optimizer, loss weighting, aggregation methods, 30-epoch training limit, early stopping, and seed 42. The only substantive modeling change is the sequence length, which increases from 31 to 61 epochs to match the TCN's theoretical receptive field. This first 61-epoch run is kept separate from later seed replication so its individual result can be reviewed before ensembling.

In [ ]:
stage16_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage16_output_dir,
    channels=CHANNELS,
    train_feature_path=train_feature_path,
    validation_feature_path=validation_feature_path,
    feature_preprocessing_metadata_path=feature_metadata_path,
    participant_array_cache_dir=array_cache_dir,
    max_train_participants=None,
    max_val_participants=None,
    random_seed=42,
)

stage16_config = None
if (
    stage15_encoder_checkpoint_path is not None
    and stage15_encoder_checkpoint_path.exists()
):
    stage16_config = build_stage16_temporal_tcn_config(
        stage15_encoder_checkpoint_path,
        base_config=stage16_base_config,
        output_dir=stage16_output_dir,
        embedding_dir=stage15_embedding_dir,
    )

stage16_config


Set `RUN_STAGE16_EXPERIMENT = True` to train the single seed-42 61-epoch MSResCNN-MLP-TCN. Leave every earlier training flag disabled. The existing embedding cache is reused when its manifest still matches the frozen encoder and source artifacts. Outputs are written under `results/stage16_temporal_fusion_tcn_s61/`.

In [ ]:
RUN_STAGE16_EXPERIMENT = False

if stage16_config is not None and RUN_STAGE16_EXPERIMENT:
    stage16_summary = run_stage16_experiment(
        stage16_config,
        output_dir=stage16_output_dir,
    )
    display(stage16_summary.sort_values("macro_f1", ascending=False))
    print("outputs:", stage16_output_dir)
else:
    print("Stage 16 seed-42 experiment is configured but not run.")


## 61-Epoch MSResCNN-MLP-TCN Seed Replication and Equal-Weight Ensemble

After the seed-42 61-epoch MSResCNN-MLP-TCN run has completed, this section discovers its center-weighted run directory, trains the same 61-epoch configuration with predetermined seeds 43 and 44, and averages all three epoch-level probability tables equally. Seed 42 is preserved and is not retrained. Each seed and the ensemble are reported without best-seed selection.

In [ ]:
stage16_summary_path = stage16_output_dir / "experiment_summary.csv"
stage16_reference_run_dir = None

if stage16_summary_path.exists():
    completed_stage16 = pd.read_csv(stage16_summary_path)
    completed_stage16_primary = completed_stage16[
        completed_stage16["aggregation_method"] == "center_weighted"
    ].sort_values(
        ["macro_f1", "balanced_accuracy"],
        ascending=[False, False],
    )
    if not completed_stage16_primary.empty:
        stage16_reference_run_dir = Path(
            completed_stage16_primary.iloc[0]["output_dir"]
        )

stage16_replication_configs = []
if stage16_config is not None and stage16_reference_run_dir is not None:
    stage16_replication_configs = build_stage16_seed_replication_configs(
        stage16_config,
        output_dir=stage16_replication_output_dir,
        seeds=(43, 44),
    )

[(config.random_seed, config.sequence_length) for config in stage16_replication_configs]


Set `RUN_STAGE16_SEED_REPLICATIONS = True` only after the seed-42 61-epoch MSResCNN-MLP-TCN run is complete. This trains seeds 43 and 44, reuses any completed replica artifacts, and creates the equal-weight seed-42/43/44 ensemble under `results/stage16_temporal_fusion_tcn_s61_seed_replication/`.

In [ ]:
RUN_STAGE16_SEED_REPLICATIONS = False

if (
    stage16_reference_run_dir is not None
    and stage16_replication_configs
    and RUN_STAGE16_SEED_REPLICATIONS
):
    stage16_seed_ensemble_summary = run_stage16_seed_replications(
        stage16_replication_configs,
        reference_run_dir=stage16_reference_run_dir,
        output_dir=stage16_replication_output_dir,
    )
    display(stage16_seed_ensemble_summary)
    print("outputs:", stage16_replication_output_dir)
else:
    print("Stage 16 seed replications are configured but not run.")
